# Stage 4.01 — setup and pinned OpenVLA-OFT policy preflight
Creates an isolated Python 3.10 environment and downloads the exact frozen checkpoint. The first run is slow and uses substantial disk space.

In [ ]:
import json,os,shutil,subprocess
from pathlib import Path
GPU="0"  # change to one idle physical A100
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; OFT=Path.home()/"openvla-oft"; LERO=Path.home()/"lerobot-stage4"; ENV=Path.home()/"venv-stage4-openvla"; PY=ENV/"bin/python"; OUT=Path.home()/"stage4"; OUT.mkdir(exist_ok=True)
OFT_SHA="e4287e94541f459edc4feabc4e181f537cd569a8"; LEROBOT_SHA="2aba372b4e217cc47db28e0f836859b20d1456c9"; CKPT="moojink/openvla-7b-oft-finetuned-libero-spatial-object-goal-10"; REV="13cdacd486c504e65408fc3c9e12fec9c5bf0382"
for path in (R,P,P/"libero/libero/assets"):
    if not path.exists(): raise SystemExit(f"STOP: missing prerequisite {path}")
line=subprocess.run(["nvidia-smi","-i",GPU,"--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print(line)
name,total,used,util,driver=[x.strip() for x in line.split(',')]; assert "A100" in name
if int(used)>=500 or int(util)>=5: raise SystemExit(f"STOP: physical GPU {GPU} is not idle: {line}")
def pinned_checkout(path,url,sha):
    if not path.exists(): subprocess.run(["git","clone",url,str(path)],check=True)
    actual=subprocess.run(["git","-C",str(path),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
    if actual!=sha:
        dirty=subprocess.run(["git","-C",str(path),"status","--porcelain"],capture_output=True,text=True,check=True).stdout
        if dirty: raise SystemExit(f"STOP: {path} is modified; preserve it before checkout")
        subprocess.run(["git","-C",str(path),"fetch","origin",sha],check=True); subprocess.run(["git","-C",str(path),"checkout","--detach",sha],check=True)
pinned_checkout(OFT,"https://github.com/moojink/openvla-oft.git",OFT_SHA)
pinned_checkout(LERO,"https://github.com/huggingface/lerobot.git",LEROBOT_SHA)
if not PY.exists():
    py310=shutil.which("python3.10"); conda=shutil.which("conda") or shutil.which("mamba")
    if py310: subprocess.run([py310,"-m","venv",str(ENV)],check=True)
    elif conda: subprocess.run([conda,"create","-y","-p",str(ENV),"python=3.10.14","pip"],check=True)
    else: raise SystemExit("STOP: Python 3.10 or conda/mamba is required")
print("PASS: pinned checkouts and Python 3.10 environment ready")

In [ ]:
marker=ENV/".stage4_dependencies_complete"
if not marker.exists():
    subprocess.run([str(PY),"-m","pip","install","--upgrade","pip","setuptools","wheel"],check=True)
    subprocess.run([str(PY),"-m","pip","install","-e",str(OFT)],check=True)
    subprocess.run([str(PY),"-m","pip","install","--no-deps","-e",str(LERO)],check=True)
    subprocess.run([str(PY),"-m","pip","install","-e",str(R),"pandas","pyarrow","pytest","gymnasium","hf-libero"],check=True)
    marker.write_text("complete\n")
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(OFT)+os.pathsep+str(R)})
versions=subprocess.run([str(PY),"-c","import sys,torch,transformers; print(sys.version.split()[0],torch.__version__,transformers.__version__)"],capture_output=True,text=True,check=True,env=env).stdout.strip(); print("runtime",versions)
pyver,torchver,transformersver=versions.split(); assert pyver.startswith("3.10.") and torchver.startswith("2.2.0") and transformersver.startswith("4.40.1")
subprocess.run([str(PY),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=env,check=True)
code=f"from huggingface_hub import snapshot_download; print(snapshot_download(repo_id='{CKPT}',revision='{REV}'))"
snapshot=subprocess.run([str(PY),"-c",code],env=env,capture_output=True,text=True,check=True).stdout.strip().splitlines()[-1]; SNAP=Path(snapshot); (OUT/"stage4_checkpoint_snapshot.txt").write_text(str(SNAP)+"\n")
required=["config.json","dataset_statistics.json","action_head--300000_checkpoint.pt","proprio_projector--300000_checkpoint.pt"]; missing=[x for x in required if not (SNAP/x).is_file()]
if missing: raise SystemExit(f"STOP: pinned checkpoint snapshot missing {missing}")
packages=subprocess.run([str(PY),"-m","pip","freeze"],capture_output=True,text=True,check=True).stdout.splitlines()
prov={"repository_sha":subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"libero_plus_sha":subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"lerobot_sha":LEROBOT_SHA,"openvla_oft_sha":OFT_SHA,"checkpoint_id":CKPT,"checkpoint_revision":REV,"checkpoint_snapshot":str(SNAP),"gpu_index":GPU,"gpu":line,"python":pyver,"packages":packages}
(OUT/"stage4_preflight_environment.json").write_text(json.dumps(prov,indent=2)+"\n"); print("PASS: Stage 4 pinned OpenVLA-OFT preflight complete")